# Test références Python: environnements multiples

Le pattern singleton a été retiré de la classe Environment: plusieurs instances peuvent désormais coexister. Chaque agent aura à terme:
- l'Environment global, partagé par tous les agents;
- un environnement local (zone de confiance): ce que l'agent peut percevoir. Il pourra exister plusieurs types de zones de confiance selon le type d'agent (hors périmètre de ce stage).

Si le même objet Agent est enregistré dans plusieurs instances d'Environment, une mise à jour de son dataset local est-elle visible depuis toutes les instances?

Scénarios testés:
1. L'agent x estime une valeur manquante et met à jour son dataset local, visible depuis l'env. global? Depuis les env. locaux des autres agents qui perçoivent x?
2. L'agent y (présent dans l'env. local de x) estime une valeur en coopérant, puis met à jour son dataset local, visible depuis l'env. global? Depuis l'env. local de x?

Piège identifié: register_agent écrase agent.environment. Ce comportement est démontré et documenté ci-dessous (section 2).

In [15]:
# Uncomment if dependencies are not installed
# %pip install -r requirements.txt

In [16]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from datetime import datetime, timedelta
import copy

from hybridiot import Agent, Environment, Position

In [17]:
# Environnement global + 3 agents
# (plus de singleton: chaque appel à Environment() crée une instance distincte)
global_env = Environment(base_dataset=None)

# Positions fictives mais réalistes (adapter la signature de Position si besoin)
pos_x = Position(latitude=43.6045, longitude=1.4440)   
pos_y = Position(latitude=43.5605, longitude=1.4680)   
pos_z = Position(latitude=43.6350, longitude=1.3740)   
agent_x = Agent(id='station_x', position=pos_x, data_type='temperature')
agent_y = Agent(id='station_y', position=pos_y, data_type='temperature')
agent_z = Agent(id='station_z', position=pos_z, data_type='temperature')

for a in (agent_x, agent_y, agent_z):
    global_env.register_agent(a)

# Quelques valeurs initiales
t0 = datetime(2020, 7, 1, 12, 0)
for i in range(4):
    ts = t0 + timedelta(minutes=15 * i)
    agent_x.add_value(value=20.0 + i, timestamp=ts)
    agent_y.add_value(value=18.0 + i, timestamp=ts)
    agent_z.add_value(value=22.0 + i, timestamp=ts)

print('global_env :', global_env)
print('Deux instances distinctes?', Environment() is not Environment())

global_env : Environment(agents=3, base_dataset='None')
Deux instances distinctes? True


## 1. Environnements locaux comme instances réelles d'Environment

Le singleton retiré, l'environnement local peut être une vraie instance d'Environment. Les agents y sont insérés par référence (mêmes objets que dans le global), sans copie, sans reconstruction.

In [18]:
# Env local de x: contient y
# Env local de y: contient x 
# Env local de z: contient x
agent_x.local_environment = Environment(base_dataset=None)
agent_y.local_environment = Environment(base_dataset=None)
agent_z.local_environment = Environment(base_dataset=None)

# Insertion par référence directe dans la liste (pas register_agent (section 2))
agent_x.local_environment.add_agent(agent_y)
agent_y.local_environment.add_agent(agent_x)
agent_z.local_environment.add_agent(agent_x)

print('local_env_x :', [a.id for a in agent_x.local_environment.agents])
print('local_env_y :', [a.id for a in agent_y.local_environment.agents])
print('local_env_z :', [a.id for a in agent_z.local_environment.agents])

local_env_x : ['station_y']
local_env_y : ['station_x']
local_env_z : ['station_x']


In [19]:
# Vérification d'identité: même objet en mémoire?
x_via_global  = global_env.get_agent('station_x', data_type='temperature')
x_via_local_y = agent_y.local_environment.get_agent('station_x', data_type='temperature')
x_via_local_z = agent_z.local_environment.get_agent('station_x', data_type='temperature')

print('x (global) is x (local de y) :', x_via_global is x_via_local_y)
print('x (global) is x (local de z) :', x_via_global is x_via_local_z)
print('id global :', id(x_via_global))
print('id local y:', id(x_via_local_y))
print('id local z:', id(x_via_local_z))

x (global) is x (local de y) : True
x (global) is x (local de z) : True
id global : 140197760993136
id local y: 140197760993136
id local z: 140197760993136


## 2. register_agent vs add_agent: deux opérations distinctes

register_agent revendique l'agent: il fait agent.environment = self et est réservé à l'environnement global. add_agent insère l'agent par référence seulement, sans toucher agent.environment, c'est la méthode à utiliser pour peupler les zones de confiance. Cette distinction résout le piège identifié précédemment (l'écrasement de la référence globale lors d'un second enregistrement).

In [ ]:
# register_agent revendique; add_agent référence seulement
print('agent_x.environment is global_env ?', agent_x.environment is global_env)

demo_env = Environment(base_dataset=None)
demo_env.add_agent(agent_x)   # ne touche PAS agent_x.environment

print('Après add_agent dans demo_env :', agent_x.environment is global_env)
print('agent_x présent dans demo_env ?', agent_x in demo_env.agents)

print('\nadd_agent insère par référence sans revendiquer l\'agent.')

agent_x.environment is global_env ? True
Après add_agent dans demo_env : True
agent_x présent dans demo_env ? True

add_agent insère par référence sans revendiquer l'agent.


## 3. Scénario 1: x estime une valeur et met à jour son dataset local

On ajoute une valeur estimée via add_value.
Attendu: la valeur est visible depuis le global et depuis les env. locaux de y et z, car toutes les instances référencent le même objet.

In [21]:
taille_avant = len(agent_x.dataset)

# x estime 24.5 au prochain pas de temps et met à jour SON dataset
ts_estime = t0 + timedelta(minutes=15 * 4)
agent_x.add_value(value=24.5, timestamp=ts_estime, is_estimate=True)

print(f'Taille dataset de x : {taille_avant} -> {len(agent_x.dataset)}')
print('Visible via GLOBAL :', len(global_env.get_agent("station_x", data_type="temperature").dataset))
print('Visible via local de y :', len(agent_y.local_environment.get_agent("station_x", data_type="temperature").dataset))
print('Visible via local de z :', len(agent_z.local_environment.get_agent("station_x", data_type="temperature").dataset))
print('Marquée estimée ? :', agent_x.is_estimated(ts_estime))

print('\nScénario 1: la mise à jour est visible depuis toutes les instances (références partagées)')

Taille dataset de x : 4 -> 5
Visible via GLOBAL : 5
Visible via local de y : 5
Visible via local de z : 5
Marquée estimée ? : True

Scénario 1: la mise à jour est visible depuis toutes les instances (références partagées)


## 4. Scénario 2: y estime en coopérant avec x, puis met à jour son dataset

y demande des fenêtres via l'Environment (env. global avec ask_data_window()), calcule une estimation, et l'ajoute à son dataset local.

Attendu: symétrique au scénario 1, visible depuis le global et depuis l'env. local de x (qui contient y).

Nuance: les DataWindow retournées par ask_data_window sont des instantanés construits au moment de la requête. «Visible» = «la prochaine lecture voit la nouvelle valeur», pas «les fenêtres déjà distribuées se mettent à jour».

In [22]:
taille_avant_y = len(agent_y.dataset)

# 1. y collecte les fenêtres des collaborateurs via l'environnement global
windows = global_env.ask_data_window(requesting_agent=agent_y)
for agent_id, dw in windows.items():
    print(f'Fenêtre reçue de {agent_id} : {dw.valeurs.shape}')

# 2. estimation: moyenne des valeurs reçues
valeurs = []
for dw in windows.values():
    valeurs.extend(dw.valeurs.flatten().tolist())
estimation = sum(valeurs) / len(valeurs) if valeurs else 0.0
print(f'\nEstimation de y : {estimation:.2f}')

# 3. y met à jour son dataset local
agent_y.add_value(value=estimation, timestamp=ts_estime, is_estimate=True)

print(f'Taille dataset de y : {taille_avant_y} -> {len(agent_y.dataset)}')
print('Visible via GLOBAL :', len(global_env.get_agent("station_y", data_type="temperature").dataset))
print('Visible via local de x :', len(agent_x.local_environment.get_agent("station_y", data_type="temperature").dataset))

print('\nScénario 2: la mise à jour de y est visible depuis le global et depuis l\'env. local de x')

Fenêtre reçue de station_z : torch.Size([1, 4, 1])
Fenêtre reçue de station_x : torch.Size([1, 5, 1])

Estimation de y : 22.72
Taille dataset de y : 4 -> 5
Visible via GLOBAL : 5
Visible via local de x : 5

Scénario 2: la mise à jour de y est visible depuis le global et depuis l'env. local de x


## 5. Contre-exemples: quand le partage NE fonctionne PAS

Deux façons de casser le partage de références. À connaître pour les éviter lors de la construction des environnements locaux.

In [23]:
# Contre-exemple A: deepcopy, objets isolés 
env_copie = Environment(base_dataset=None)
env_copie.agents.append(copy.deepcopy(agent_x))

taille_avant = len(agent_x.dataset)
env_copie.agents[0].add_value(value=99.9, timestamp=ts_estime + timedelta(minutes=15))

print('Dataset de x (global) :', len(agent_x.dataset))
print('Dataset de x (copie) :', len(env_copie.agents[0].dataset))
print('Même objet ?', env_copie.agents[0] is agent_x)

print('\ndeepcopy: mutations invisibles depuis le global (isolement)')

Dataset de x (global) : 5
Dataset de x (copie) : 6
Même objet ? False

deepcopy: mutations invisibles depuis le global (isolement)


In [24]:
# Contre-exemple B: reconstruction par erreur, même id logique, état VIDE
# Si quelqu'un "filtre" en recréant des Agent au lieu de réutiliser les références
env_bug = Environment(base_dataset=None)
env_bug.agents.append(
    Agent(id=agent_x.id, position=agent_x.position, data_type=agent_x.data_type)
)

print('Même id logique ? ', env_bug.agents[0].id == agent_x.id)
print('Même objet ? ', env_bug.agents[0] is agent_x)
print('Dataset original :', len(agent_x.dataset), 'valeurs')
print('Dataset reconstruit :', len(env_bug.agents[0].dataset), 'valeurs, historique PERDU')

print('\nReconstruction: identité logique (id) != identité d\'objet (is)')

Même id logique ?  True
Même objet ?  False
Dataset original : 5 valeurs
Dataset reconstruit : 0 valeurs, historique PERDU

Reconstruction: identité logique (id) != identité d'objet (is)
